# Copy, Move and Delete

Copy and move a file or a whole folder, change the metadata of an object while copying it, and delete files and folders. The copies happen inside S3: no data goes through your machine.

## Setup

This cell only prepares the notebook: it connects to the S3 bucket that the s3pathlib CI uses and empties the folder of this chapter, so every run starts from the same state. In your own code you only need the two lines marked below, with your own AWS profile and bucket.
`show_error` prints an exception in one line, so the demos can show what goes wrong.

In [1]:
import os

import boto3
from s3pathlib.api import S3Path
from s3pathlib.api import s3_context

boto_ses = boto3.Session(profile_name=os.environ["S3PATHLIB_TEST_AWS_PROFILE"])
s3_context.attach_s3_client(boto_ses.client("s3"))  # <-- in your code
aws_account_id = boto_ses.client("sts").get_caller_identity()["Account"]
bucket = f"{aws_account_id}-{boto_ses.region_name}-s3pathlib-test"  # <-- your bucket

# every demo of this chapter writes under this folder
s3dir_chapter = S3Path(bucket, "projects/s3pathlib/docs/06-Copy-Move-Delete/")
_ = s3dir_chapter.delete()


def show_error(func):
    try:
        func()
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

## 1. Copy a file

`copy_file(dst)` copies one object and returns the new `S3Path` of the copy, with the ETag and last modified time from the copy call. It never replaces an existing object unless you pass `overwrite=True`; on AWS the copy is conditional, so an object written by someone else at the same moment is not replaced either.

In [2]:
s3dir = s3dir_chapter / "copy-file/"
src = s3dir / "src.txt"
_ = src.write_text("hello", metadata={"author": "alice"})

dst = s3dir / "dst.txt"
copied = src.copy_file(dst)
print(copied - s3dir, copied.etag == src.etag)
print(dst.read_text(), dst.metadata)

S3RelPath('dst.txt') True
hello {'author': 'alice'}


In [3]:
show_error(lambda: src.copy_file(dst))
_ = src.copy_file(dst, overwrite=True)
show_error(lambda: (s3dir / "missing.txt").copy_file(dst, overwrite=True))

S3FileAlreadyExist: S3 object 's3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/06-Copy-Move-Delete/copy-file/dst.txt' already exists!
S3FileNotExist: S3 object 's3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/06-Copy-Move-Delete/copy-file/missing.txt' does not exist!


The target can be in another bucket, and `s3_client=...` copies with another client (e.g. another account with access to both buckets). Copying with `copy_object` works up to 5 GB per object; for a bigger one S3 raises `InvalidRequest`, use boto3's managed `s3_client.copy()`.

## 2. Change metadata and headers while copying

By default the copy keeps the user metadata, the tags and the headers (`content_type`, `cache_control`, ...) of the source. Pass any of them to replace **only** that one. (S3 itself would reset the headers you did not pass; s3pathlib reads them from the source and sends them again.)

In [4]:
src = s3dir / "page.html"
_ = src.write_text("<h1>Hi</h1>", content_type="text/html", metadata={"author": "alice"})

dst = s3dir / "page-copy.html"
_ = src.copy_file(dst, metadata={"author": "bob"}, tags={"copied": "yes"})
dst.head_object()["ContentType"], dst.metadata, dst.get_tags()[1]

('text/html', {'author': 'bob'}, {'copied': 'yes'})

S3 cannot edit the metadata of an object. Copying the object onto itself with new metadata is how you change it; the content is not uploaded again.

In [5]:
_ = src.copy_file(src, metadata={"author": "carol"}, overwrite=True)
src.head_object()["ContentType"], src.metadata, src.read_text()

('text/html', {'author': 'carol'}, '<h1>Hi</h1>')

## 3. Copy a folder

`copy_dir(dst)` copies every object under a folder, folder markers included, keeping the paths relative to the folder, and returns how many it copied. With `overwrite=False` (the default) it checks every target first and copies nothing if one exists.

In [6]:
s3dir = s3dir_chapter / "copy-dir/"
src_dir = s3dir / "src/"
for key in ["a.txt", "sub/b.txt"]:
    _ = (src_dir / key).write_text(key)
(src_dir / "empty/").mkdir()

dst_dir = s3dir / "dst/"
print(src_dir.copy_dir(dst_dir))
print([p - dst_dir for p in dst_dir.iter_objects()])
print((dst_dir / "empty/").exists())

3
[S3RelPath('a.txt'), S3RelPath('sub/b.txt')]
True


The 3 copied keys are the two files and the folder marker of `empty/`, which `iter_objects` does not list.

The two folders must not overlap: copying a folder into itself would copy objects that are being copied.

In [7]:
show_error(lambda: src_dir.copy_dir(dst_dir, overwrite=False))
show_error(lambda: src_dir.copy_dir(src_dir / "backup/"))

S3FileAlreadyExist: S3 object 's3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/06-Copy-Move-Delete/copy-dir/dst/a.txt' already exists!
ValueError: 's3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/06-Copy-Move-Delete/copy-dir/src/' and 's3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/06-Copy-Move-Delete/copy-dir/src/backup/' overlap: copying would overwrite source objects before they are copied (and moving would delete the copies); copy to another folder first


`copy_to` does either: `copy_file` for a file, `copy_dir` for a folder, and returns the number of objects copied.

In [8]:
print((src_dir / "a.txt").copy_to(s3dir / "a-copy.txt"))
print(src_dir.copy_to(s3dir / "dst-2/"))

1
3


## 4. Move

`move_to` copies, then deletes exactly the source keys it copied. It is not atomic: if it fails half way, some objects are in both places.

In [9]:
s3dir = s3dir_chapter / "move/"
_ = (s3dir / "inbox/1.txt").write_text("1")
_ = (s3dir / "inbox/2.txt").write_text("2")

print((s3dir / "inbox/").move_to(s3dir / "archive/"))
print([p - s3dir for p in s3dir.iter_objects()])

2
[S3RelPath('archive/1.txt'), S3RelPath('archive/2.txt')]


In [10]:
print((s3dir / "archive/1.txt").move_to(s3dir / "1.txt"))
print([p - s3dir for p in s3dir.iter_objects()])
show_error(lambda: (s3dir / "1.txt").move_to(s3dir / "1.txt"))

1
[S3RelPath('1.txt'), S3RelPath('archive/2.txt')]
ValueError: cannot move 's3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/06-Copy-Move-Delete/move/1.txt' to itself


## 5. Delete

`delete()` deletes a file, or every object under a folder (folder markers included). Deleting what does not exist is not an error. It returns a new `S3Path`; in a bucket with versioning, that path is the new delete marker, see [Versioning](../07-Versioning/index.ipynb).

In [11]:
s3dir = s3dir_chapter / "delete/"
for key in ["a.txt", "sub/b.txt", "sub/c.txt"]:
    _ = (s3dir / key).write_text(key)

_ = (s3dir / "a.txt").delete()
print([p - s3dir for p in s3dir.iter_objects()])
_ = (s3dir / "sub/").delete()
print(s3dir.exists())
_ = (s3dir / "missing.txt").delete()

[S3RelPath('sub/b.txt'), S3RelPath('sub/c.txt')]
False


Deleting a bucket path deletes **everything** in the bucket, so it first asks you to type `YES` (pass `skip_prompt=True` in scripts). Deleting the bucket itself is `delete_bucket()`, see [Buckets](../08-Buckets/index.ipynb).

## Common pitfalls

- **Overwrite is off by default.** `copy_file`, `copy_dir`, `copy_to`, `move_to` and `upload_*` raise `S3FileAlreadyExist` instead of replacing; pass `overwrite=True`.
- **Moving is copy plus delete.** It is not atomic, and on a large folder it takes as long as a copy.
- **Objects over 5 GB** cannot be copied with these methods; use boto3's `s3_client.copy()`.
- **Delete without versioning is final.** Turn on versioning in the bucket if you need to undo deletes.

## See also

- [Metadata and Tags](../05-Metadata-and-Tags/index.ipynb): what a copied path knows about its object.
- [Versioning](../07-Versioning/index.ipynb): copy an older version (`version_id`), delete markers, permanent deletes.
- [Sync](../09-Sync/index.ipynb): copy only what changed between two folders.